In [1]:
from sympy import symbols, Eq, solve
import re
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

token = os.environ.get("HUGGING_TOKEN")

In [3]:
from transformers import pipeline

In [4]:
generator = pipeline("text-generation", model="mistralai/Mistral-7B-Instruct-v0.3", token=token, temperature=0.001)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Device set to use cuda:0


In [39]:
prompt = "2 times x plus 20 y equals 10. 10x minus five y equals 0. z equals x to the power of 3"
instruction = [
    {
        "role": "system", "content":
            "You are a natural language equation parser. You will receive an equation described in a natural language.\n"
            "1. Output ONLY a comma-separated list of equations. Each equation must:\n"
            "   - Be in single quotes: 'example'\n"
            "   - Consist of two expressions separated by =\n"
            "   - Both of the expressions must consist of named variables, numbers, and operators between them\n"
            "   - The named variables consist of latin characters only, e.g. x, y, var, john, apple etc.\n"
            "   - The numbers should use '.' for decimal points when necessary\n"
            "   - The only operators allowed are: +, -, *, /, ** and there should be spaces on both sides of each operator \n"
            "2. If the input describes an inequality (>, <, >=, <=, !=, or their verbal forms), respond ONLY with: INEQUAL_WARNING.\n"
            "3. If the input does not describe a valid math equation, respond ONLY with: NOTMATH_WARNING.\n"
            "Do not solve the equations. Do not explain anything. Do not output code. Output nothing except what the rules above require."
    },
]

messages = instruction + [
  {"role": "user", "content": "This is the equation described in a natural language:\n<<<\n3 times a plus 4b equals 7\n>>>"},
  {"role": "assistant", "content": "'3 * a + 4 * b = 7'"},
  {"role": "user", "content": "This is the equation described in a natural language:\n<<<\ntwo a minus twentyone equals b. and c squared equals b as well. c=2a\n>>>"},
  {"role": "assistant", "content": "'2 * a - 21 = b', 'c ** 2 = b', c = 2 * a"},
  {"role": "user", "content": "This is the equation described in a natural language:\n<<<\n2x minus five equals zero\n>>>"},
  {"role": "assistant", "content": "'2 * x - 5 = 0'"},
  {"role": "user", "content":
    f"This is the equation described in a natural language:\n<<<\n{prompt}\n>>>"
  },
]

In [40]:
result = generator(messages)


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


In [41]:
def result_parser (result) :
    answer = result[0]["generated_text"] [-1] ['content']

    answers = answer.split(', ')
    answers_cleared = []

    for elem in answers :
        elem_clr = elem.replace("'", "")

        answers_cleared.append(elem_clr)

    return answers_cleared


In [42]:
def explicit_multiplication (equation) :
    import re

    pattern = r'([0-9])\s*([A-Za-z])|([A-Za-z])\s*([0-9])'

    def insert_multiply(match):
        if match.group(1) and match.group(2):
            # digit+letter
            return f"{match.group(1)} * {match.group(2)}"
        else:
            # letter+digit
            return f"{match.group(3)} * {match.group(4)}"

    eq_expl = re.sub(pattern, insert_multiply, equation)
    return eq_expl

In [43]:
parsed_answer = result_parser(result)

equations = []
for eq in parsed_answer :
    equations.append(explicit_multiplication(eq))

print(equations)

[' 2 * x + 20 * y = 10', '10 * x - 5 * y = 0', 'z = x ** 3']


In [44]:
import re

def is_safe_equation(s: str) -> bool:
    if s.count('=') != 1: return False # only one equation sign
    if re.search(r"[\"'`_<>!^&|:%,$\\\[\]{}]", s): return False # forbidden chars
    if not re.fullmatch(r"[A-Za-z0-9+\-*/=().\s]+", s): return False # allowed chars
    if re.search(r"[A-Za-z]\s*\.\s*[A-Za-z0-9]", s): return False # a.b not allowed
    if re.search(r"[A-Za-z][A-Za-z0-9]*\s*\(", s): return False # not allowed fun(
    L, R = (p.strip() for p in s.split('='))
    if not L or not R: return False # something on both sides of equation

    bal = 0
    for ch in s: # both brackets present
        bal += (ch == '(') - (ch == ')')
        if bal < 0: return False
    return bal == 0

In [45]:
for eq in equations :
    if not is_safe_equation(eq) :
        raise Exception(f"The equation {eq} is not safe.")

In [46]:
symbol_names = set()
for eq in equations :
    matches = re.findall(r"[A-Za-z]+", eq)
    symbol_names.update(matches)

print(f'Symbol names: {symbol_names}')

Symbol names: {'y', 'z', 'x'}


In [47]:
from sympy import symbols

sympy_symbols = symbols(" ".join(symbol_names))

In [48]:
from sympy import sympify
from sympy import Eq

equation_set = []

for eq in equations :
    left, right = eq.split('=')

    equation_set.append(
        Eq(sympify(left), sympify(right))
    )

In [49]:
solve(equation_set, sympy_symbols, dict=True)

[{x: 5/21, y: 10/21, z: 125/9261}]